# Chapter 5: join a sample to a batch without guessing

Section 5.5 ranks three joins by trust: the fill ID on the label, the vessel and the sampled time, then the
vessel and the date. This notebook runs all three on the synthetic sample table and reports the share matched
by each method.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import kestrel as k

raw = k.load("bronze_lims_sample")
batch = k.load("batch")
vessel = k.load("vessel")
events = k.load("vessel_event")
print(raw.to_string(index=False))

Occupancy: when each vessel held a batch. A fermenter holds its fill from the first fill to empty. A bright
tank holds the beer from its bright fill for up to six days at Kestrel (a working value).

In [ ]:
IST = pd.Timedelta(hours=5, minutes=30)
BRIGHT_HOLD = pd.Timedelta(days=6)
codes = vessel.drop_duplicates("vessel_code", keep="last").set_index("vessel_key").vessel_code
occ = batch.assign(vessel_code=batch.vessel_key.map(codes), start=pd.to_datetime(batch.fill_start_utc),
                   end=pd.to_datetime(batch.empty_utc))[["vessel_code", "start", "end", "batch_id"]]
bright = events[events.event_type == "bright fill"].merge(batch[["batch_key", "batch_id"]], on="batch_key")
bright = bright.assign(vessel_code=bright.vessel_key.map(codes), start=pd.to_datetime(bright.start_utc))
bright["end"] = bright.start + BRIGHT_HOLD
occ = pd.concat([occ, bright[["vessel_code", "start", "end", "batch_id"]]], ignore_index=True)

In [ ]:
def join(row):
    if isinstance(row.label_id, str) and row.label_id.startswith("F"):
        return row.label_id, "label"
    here = occ[occ.vessel_code == row.vessel_code]
    if isinstance(row.sampled_time_ist, str):
        t = pd.Timestamp(f"{row.sampled_date_ist} {row.sampled_time_ist}") - IST
        hit = here[(here.start <= t) & (t <= here.end)]
        return (hit.batch_id.iloc[0], "vessel_time") if len(hit) else (None, "none")
    day = pd.Timestamp(row.sampled_date_ist).date()
    hit = here[(here.start.dt.date <= day) & (day <= here.end.dt.date)]
    return (hit.batch_id.iloc[0], "vessel_date") if len(hit) else (None, "none")

raw[["batch_id", "match_method"]] = raw.apply(join, axis=1, result_type="expand")
print(raw[["sample_id", "vessel_code", "label_id", "batch_id", "match_method"]].to_string(index=False))

The share matched by each method. A dashboard can show this beside every number that rests on a join.

In [ ]:
share = raw.match_method.value_counts(normalize=True).mul(100).round(1)
print(share.to_string())
gold = k.load("sample")
assert (gold.match_method.values == raw.match_method.values).all(), "differs from the gold sample table"
print("matches the match_method column in the gold sample table")

The vessel and date match is weak: use it only to flag a result for a person. The sample with no match came
from FV10 while it was empty, so it goes to quarantine with the reason "no event" (chapter 12).